# SS_06 - GBM IDH Classifier - Interfaz Streamlit
Ejecutar las celdas en orden.

In [1]:
# CELDA 1 - Instalacion de dependencias
!pip install streamlit monai[all] nibabel scipy numpy pandas joblib scikit-learn lightgbm pydicom SimpleITK -q
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git -q
!pip install hd-bet -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 158.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/6

In [2]:
# CELDA 2 - Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# CELDA 3 - Restaurar pesos HD-BET desde Drive
import shutil
from pathlib import Path

hdbet_drive = Path('/content/drive/MyDrive/GBM_TESIS/hdbet_weights')
hdbet_cache = Path('/root/hd-bet_params')

if hdbet_drive.exists() and not hdbet_cache.exists():
    shutil.copytree(str(hdbet_drive), str(hdbet_cache), dirs_exist_ok=True)
    print('Pesos HD-BET restaurados desde Drive.')
else:
    print('Pesos HD-BET ya disponibles.')

Pesos HD-BET restaurados desde Drive.


In [4]:
# CELDA 4 - Copiar archivos desde Drive a Colab
from pathlib import Path

BASE      = Path('/content/drive/MyDrive/GBM_TESIS')
INTERFAZ  = BASE / 'Interfaz'

Path('/content/app.py').write_text(
    (INTERFAZ / 'app.py').read_text(encoding='utf-8'), encoding='utf-8')

Path('/content/gbm_pipeline.py').write_text(
    (INTERFAZ / 'gbm_pipeline.py').read_text(encoding='utf-8'), encoding='utf-8')

print('Archivos copiados desde Drive/Interfaz/.')


Archivos copiados desde Drive/Interfaz/.


In [ ]:
# CELDA 5 - Lanzar Streamlit con cloudflared
import subprocess
import time
import re

# Matar procesos anteriores
subprocess.run(['pkill', '-9', '-f', 'streamlit'], capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], capture_output=True)
subprocess.run(['fuser', '-k', '8501/tcp'], capture_output=True)
time.sleep(2)

# Descargar cloudflared
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/tmp/cloudflared'
], capture_output=True)
subprocess.run(['chmod', '+x', '/tmp/cloudflared'], capture_output=True)

# Lanzar Streamlit
proc = subprocess.Popen(
    ['streamlit', 'run', '/content/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
)

time.sleep(6)

# Lanzar tunel cloudflared
tunnel = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print('Esperando URL del tunel...')
for line in tunnel.stdout:
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        print(f'\nInterfaz disponible en: {match.group(0)}\n')
        break

try:
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    tunnel.terminate()
    print('Streamlit detenido.')

Esperando URL del tunel...

Interfaz disponible en: https://drainage-oregon-independently-tall.trycloudflare.com

